In [1]:
import dgl.nn as dglnn
import dgl
from dgl import from_networkx
import torch.nn as nn
import torch as th
import torch.nn.functional as F
import dgl.function as fn
import networkx as nx
import pandas as pd
import socket
import struct
import random
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import category_encoders as ce
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [2]:
device = th.device('cuda' if th.cuda.is_available() else 'cpu')
graphs, _ = dgl.load_graphs('train_m.bin')
G = graphs[0]

In [3]:
G = G.to(device)
node_features = G.ndata['h']
edge_features = G.edata['h']
edge_label = G.edata['label']


In [4]:
print(node_features.shape[0])
print(edge_features.shape[0])

434536
711502


In [5]:
def compute_accuracy(pred, labels):
    return (pred.argmax(1) == labels).float().mean().item()

class SAGELayer(nn.Module):
    def __init__(self, ndim_in, edims, ndim_out ):
        super(SAGELayer, self).__init__()
        ### force to outut fix dimensions
        self.W_msg = nn.Linear(ndim_in + edims, ndim_out)
        ### apply weight
        self.W_apply = nn.Linear(ndim_in + ndim_out, ndim_out)
        

    def message_func(self, edges):
        return {'m': self.W_msg(th.cat([edges.data['h'], edges.dst['h']], 1))}

    def forward(self, g_dgl, nfeats, efeats):
        with g_dgl.local_scope():
            g = g_dgl
            g.ndata['h'] = nfeats
            g.edata['h'] = efeats
            # Eq4
            g.update_all(self.message_func, fn.mean('m', 'h_neigh'))
            # Eq5          
            g.ndata['h'] = F.relu(self.W_apply(th.cat([g.ndata['h'], g.ndata['h_neigh']], 1)))
            return g.ndata['h']

class MLPPredictor(nn.Module):
    def __init__(self, in_features, edim, out_classes):
        super().__init__()
        self.W = nn.Linear(in_features * 2 + edim, out_classes)

    def apply_edges(self, edges):
        h_u = edges.src['h']
        h_v = edges.dst['h']
        h_e = edges.data['h']
        score = self.W(th.cat([h_u, h_e, h_v], 1))
        return {'score': score}

    def forward(self, graph, h, efeats):
        with graph.local_scope():
            graph.ndata['h'] = h
            graph.edata['h'] = efeats
            graph.apply_edges(self.apply_edges)
            return graph.edata['score']

class Model(nn.Module):
    def __init__(self,  ndim_in, ndim_out, edim):
        super().__init__()
        self.atten = nn.Parameter(th.randn(1, edim))
        self.cov1 = SAGELayer(ndim_in, edim, ndim_out)
        self.cov2 = SAGELayer(ndim_out, edim, ndim_out)
        self.pred = MLPPredictor(ndim_out, edim, 10)
        self.dropout = nn.Dropout(p=0.2)
        
    def forward(self, g, nfeats, efeats):
        # efeats = efeats * self.atten
        nfeats = F.relu(self.cov1(g, nfeats, efeats))
        nfeats = self.dropout(nfeats)
        nfeats = F.relu(self.cov2(g, nfeats, efeats))
        nfeats = self.dropout(nfeats)
        return self.pred(g, nfeats, efeats)


In [6]:
from sklearn.utils import class_weight
class_weights = class_weight.compute_class_weight(class_weight='balanced',
                                                 classes=np.unique(G.edata['label'].cpu().numpy()),
                                                 y=G.edata['label'].cpu().numpy())

In [7]:
class_weights = th.FloatTensor(class_weights).cuda()
criterion = nn.CrossEntropyLoss(weight = class_weights)

In [8]:

model = Model( G.ndata['h'].shape[1], 128, G.edata['h'].shape[1]).to(device)
opt = th.optim.Adam(model.parameters())

In [9]:
for epoch in range(350):
    pred = model( G, node_features, edge_features)
    loss = criterion(pred ,edge_label)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if (epoch+1) % 100 == 0:
      print('Training acc:', compute_accuracy(pred, edge_label))

Training acc: 0.6528288125991821
Training acc: 0.844467043876648
Training acc: 0.8601802587509155


In [10]:
graphs, _ = dgl.load_graphs('test_m.bin')
G_test = graphs[0]

In [11]:
G_test = G_test.to(device)
node_features_test = G_test.ndata['h']
edge_features_test = G_test.edata['h']
edge_label_test = G_test.edata['label']


In [12]:
print(node_features_test.shape[0])
print(edge_features_test.shape[0])

193975
304930


In [13]:
#w-128-550-0.06
out = model( G_test, node_features_test, edge_features_test)
out = F.softmax(out, dim=1)
_, pred = out.max(dim=1)
print(classification_report(edge_label_test.cpu().numpy(), pred.cpu().numpy(), digits=4))

              precision    recall  f1-score   support

           0     0.9799    0.9225    0.9503    109790
           1     0.7824    1.0000    0.8779       302
           2     0.9561    0.9199    0.9377     36472
           3     0.4814    0.9393    0.6366     12828
           4     0.6161    0.6695    0.6417     12320
           5     0.0307    0.3000    0.0557       140
           6     0.8147    0.8276    0.8211     20760
           7     0.0516    0.9677    0.0980        62
           8     0.9311    0.9629    0.9467     68066
           9     0.9007    0.6861    0.7789     44190

    accuracy                         0.8808    304930
   macro avg     0.6545    0.8195    0.6744    304930
weighted avg     0.9069    0.8808    0.8880    304930

